In [1]:
import requests as rq
import bs4
import pandas as pd
import numpy as np
from io import StringIO

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'
page = rq.get(url, headers={"User-Agent": "Chrome"})
page.text[0 : 99]

'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-l'

In [3]:
bs4page = bs4.BeautifulSoup(page.text, 'html.parser')
tables = bs4page.find_all('table',{'class':"wikitable"})

In [4]:
gdp = pd.read_html(StringIO(str(tables[0])))[0]
with pd.option_context("display.max_rows", None): display(gdp)

,Country/Territory,IMF (2026)[1],World Bank (2025)[6],United Nations (2024)[7]
0,World,126295331,118350166,111565144
1,United States,32383920,30769700,29298000
2,China[n 1],20851593,19498039,18743802
3,Germany,5452858,5050923,4659929
4,Japan,4379253,4435163,4026211
5,United Kingdom,4264794,4002588,3685881
6,India,4153191,3956067,3952244
7,France,3596094,3366316,3160443
8,Italy,2738164,2551557,2380825
9,Russia,2656452,2561310,2173386


In [10]:
url = 'https://en.wikipedia.org/w/index.php?title=List_of_countries_by_GDP_(nominal)&oldid=1187446467'
page = rq.get(url, headers={"User-Agent": "Chrome"})

bs4page = bs4.BeautifulSoup(page.text, 'html.parser')
table = bs4page.find("caption", string="GDP (USD million) by country").find_parent("table")

data = []
for row in table.find_all("tr"):
    cells = row.find_all("td")
    if len(cells) != 8: continue
    values = [cell.get_text(" ", strip=True) for cell in cells]
    data.append({"Country": values[0], "Region": values[1],})

regions = pd.DataFrame(data)
with pd.option_context("display.max_rows", None): display(regions)

,Country,Region
0,World,—
1,United States,Americas
2,China,Asia
3,Germany,Europe
4,Japan,Asia
5,India,Asia
6,United Kingdom,Europe
7,France,Europe
8,Italy,Europe
9,Brazil,Americas


In [11]:
gdp["Country/Territory"] = (gdp["Country/Territory"].str.replace(r"\[.*?\]", "", regex=True).str.strip())
regions["Country"] = (regions["Country"].str.replace(r"\[.*?\]", "", regex=True).str.strip())
df = (gdp.merge(regions, left_on="Country/Territory",right_on="Country",how="left").drop(columns="Country"))
with pd.option_context("display.max_rows", None): display(df)

,Country/Territory,IMF (2026)[1],World Bank (2025)[6],United Nations (2024)[7],Region
0,World,126295331,118350166,111565144,—
1,United States,32383920,30769700,29298000,Americas
2,China,20851593,19498039,18743802,Asia
3,Germany,5452858,5050923,4659929,Europe
4,Japan,4379253,4435163,4026211,Asia
5,United Kingdom,4264794,4002588,3685881,Europe
6,India,4153191,3956067,3952244,Asia
7,France,3596094,3366316,3160443,Europe
8,Italy,2738164,2551557,2380825,Europe
9,Russia,2656452,2561310,2173386,Europe


In [12]:
df = df.replace(["—N/a", "NaN"], np.nan)
df["GDP"] = (df["IMF (2026)[1]"].fillna(df["World Bank (2025)[6]"]).fillna(df["United Nations (2024)[7]"]))
df["GDP"] = (df["GDP"].astype(str).str.replace(r"\s*\(\d{4}\)", "", regex=True))
with pd.option_context("display.max_rows", None): display(df)

,Country/Territory,IMF (2026)[1],World Bank (2025)[6],United Nations (2024)[7],Region,GDP
0,World,126295331,118350166,111565144,—,126295331
1,United States,32383920,30769700,29298000,Americas,32383920
2,China,20851593,19498039,18743802,Asia,20851593
3,Germany,5452858,5050923,4659929,Europe,5452858
4,Japan,4379253,4435163,4026211,Asia,4379253
5,United Kingdom,4264794,4002588,3685881,Europe,4264794
6,India,4153191,3956067,3952244,Asia,4153191
7,France,3596094,3366316,3160443,Europe,3596094
8,Italy,2738164,2551557,2380825,Europe,2738164
9,Russia,2656452,2561310,2173386,Europe,2656452
